# Gemma 4 E4B Legal VLM Re-Attachment

**Goal**: Merge the GRPO-trained text LoRA adapter into the full Gemma 4 E4B base model
while **preserving** the vision encoder (~150M params) and audio encoder (~300M params).

**Output**: `gemma4-legal-vlm:latest` — one model with legal reasoning + vision + (future) audio.

## Architecture
```
google/gemma-4-e4b-it (full base)
  ├── language_model (3.9B params) ← apply GRPO LoRA adapter (588 tensors)
  ├── vision_tower (~150M params)  ← KEEP ORIGINAL (frozen, never trained)
  ├── audio_tower (~300M params)   ← KEEP ORIGINAL (frozen, never trained)
  └── multi_modal_projector        ← KEEP ORIGINAL
```

## Key Difference from GRPO Notebook
The GRPO notebook (cell 17a) **strips** vision/audio towers before merge → text-only GGUF.
This notebook **preserves** them → multimodal GGUF (language + mmproj).

## Prerequisites
- **Runtime**: G4 GPU (Blackwell 96GB) or A100 — need ~40GB for BF16 merge
- **Adapter**: `Semaj90/gemma4-e4b-legal-grpo` on HF Hub (or Google Drive)
- **HF Token**: Set in Colab Secrets as `HF_TOKEN`

## Steps
1. Install Unsloth + llama.cpp
2. Download adapter from HF Hub
3. Strip ONLY vision/audio LoRA tensors (untrained, accidental)
4. Load full base model + apply language-only LoRA
5. Dequantize → merge → save full model (with vision/audio towers intact)
6. Export multimodal GGUF: language model + mmproj
7. Test vision inference
8. Test video frame analysis
9. Package for download + Ollama deployment

## 1. Install Dependencies

In [ ]:
# Install Unsloth (must include PR #4807 ClippableLinear fix)
!pip uninstall unsloth mergekit mergekit-moe -y 2>/dev/null || true
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft transformers huggingface_hub pillow

# Pin torchao to avoid torch.int1 AttributeError (requires PyTorch 2.7+)
# If your Colab runtime has torch<2.7, torchao>=0.8 will crash on import
import torch
torch_major, torch_minor = [int(x) for x in torch.__version__.split('.')[:2]]
if torch_major < 2 or (torch_major == 2 and torch_minor < 7):
    print(f'PyTorch {torch.__version__} detected (<2.7) — pinning torchao==0.7.0')
    import subprocess
    subprocess.run(['pip', 'install', 'torchao==0.7.0', '--quiet'], check=False)
else:
    print(f'PyTorch {torch.__version__} — torchao version OK')

# Install llama.cpp for GGUF conversion (with mmproj support)
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !cd llama.cpp && pip install -r requirements.txt 2>/dev/null || true
else:
    !cd llama.cpp && git pull
    print('llama.cpp already cloned')

# Verify
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

## 2. HuggingFace Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab Secrets')
except Exception:
    print('Set HF_TOKEN in Colab Secrets (key icon in sidebar)')
    login()

## 3. Download Adapter + Strip Vision/Audio LoRA

**Two paths** — pick ONE:

### Path A: Upload pre-stripped adapter from local machine (FASTEST)
If you already have `gemma4-legal-text-only-adapter/adapter_model.safetensors` (146 MB)
in your Downloads folder, upload it directly:
1. Click the **Files** icon in Colab sidebar (📁)
2. Upload your `gemma4-legal-text-only-adapter/` folder contents
3. Set `USE_LOCAL_ADAPTER = True` in the cell below

### Path B: Download from HF Hub + strip automatically
Downloads the full 884-tensor adapter from `Semaj90/gemma4-e4b-legal-grpo`,
strips the 224 vision + 72 audio LoRA tensors (untrained, accidental),
keeps 588 language tensors.

The base model's **original** vision/audio towers remain intact — we only modify language weights.

In [ ]:
import os, json, shutil
from safetensors.torch import load_file, save_file

# ============================================================
# TOGGLE: Set True if you uploaded the pre-stripped adapter
#         from c:/Users/james/Downloads/gemma4-legal-text-only-adapter/
# ============================================================
USE_LOCAL_ADAPTER = False

ADAPTER_DIR = 'gemma4-e4b-legal-grpo-lora'
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'

if USE_LOCAL_ADAPTER:
    # ---- Path A: Use uploaded pre-stripped adapter ----
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_model.safetensors'), (
        f'Upload adapter_model.safetensors to {TEXT_ONLY_DIR}/ first!\n'
        'Local path: c:/Users/james/Downloads/gemma4-legal-text-only-adapter/'
    )
    assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_config.json'), (
        f'Upload adapter_config.json to {TEXT_ONLY_DIR}/ alongside the safetensors file!\n'
        'PEFT needs both files to load the adapter correctly.'
    )
    tensors = load_file(f'{TEXT_ONLY_DIR}/adapter_model.safetensors')
    print(f'Path A: Using uploaded pre-stripped adapter')
    print(f'  {len(tensors)} tensors ({sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2:.1f} MB)')
    vis = [k for k in tensors if 'vision_tower' in k]
    aud = [k for k in tensors if 'audio_tower' in k]
    print(f'  vision_tower: {len(vis)} (should be 0)')
    print(f'  audio_tower:  {len(aud)} (should be 0)')
    del tensors

else:
    # ---- Path B: Download from HF Hub + strip ----
    if not os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'):
        print('Downloading adapter from HF Hub...')
        from huggingface_hub import snapshot_download
        snapshot_download('Semaj90/gemma4-e4b-legal-grpo', local_dir=ADAPTER_DIR)
        print(f'Downloaded to {ADAPTER_DIR}/')
    else:
        print(f'Adapter already at {ADAPTER_DIR}/')

    # Load and inspect
    tensors = load_file(f'{ADAPTER_DIR}/adapter_model.safetensors')
    keys = sorted(tensors.keys())
    lang = {k: v for k, v in tensors.items() if 'language_model' in k}
    vis = [k for k in keys if 'vision_tower' in k]
    aud = [k for k in keys if 'audio_tower' in k]

    print(f'\nOriginal adapter: {len(keys)} tensors')
    print(f'  language_model: {len(lang)}')
    print(f'  vision_tower:   {len(vis)} (untrained — will strip LoRA only)')
    print(f'  audio_tower:    {len(aud)} (untrained — will strip LoRA only)')

    # Strip vision/audio LoRA (untrained noise) — keep language only
    os.makedirs(TEXT_ONLY_DIR, exist_ok=True)
    save_file(lang, f'{TEXT_ONLY_DIR}/adapter_model.safetensors')

    # Copy and update config
    with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
        config = json.load(f)
    config['exclude_modules'] = ['vision_tower.*', 'audio_tower.*', 'multi_modal_projector.*']
    with open(f'{TEXT_ONLY_DIR}/adapter_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    # Copy tokenizer + other files
    for fname in os.listdir(ADAPTER_DIR):
        if fname not in ('adapter_model.safetensors', 'adapter_config.json'):
            src = os.path.join(ADAPTER_DIR, fname)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(TEXT_ONLY_DIR, fname))

    orig_mb = sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2
    new_mb = sum(v.nelement() * v.element_size() for v in lang.values()) / 1024**2
    print(f'\nAdapter surgery: {len(keys)} -> {len(lang)} tensors')
    print(f'  {orig_mb:.1f} MB -> {new_mb:.1f} MB (saved {orig_mb - new_mb:.1f} MB)')
    print(f'\nLanguage LoRA saved to: {TEXT_ONLY_DIR}/')
    print('Vision/audio towers will come from the BASE MODEL (original, untouched)')

    del tensors, lang

print(f'\n=== Ready for merge ===')
print(f'Text-only adapter: {TEXT_ONLY_DIR}/adapter_model.safetensors')
print(f'Adapter config:    {TEXT_ONLY_DIR}/adapter_config.json')

## 4. Load Full Base Model + Apply Language LoRA

Key difference from GRPO notebook: we load the **full multimodal** base model.
The vision/audio towers remain frozen with their original pretrained weights.
Only language_model weights get the GRPO LoRA applied.

In [ ]:
import torch
from unsloth import FastVisionModel, is_bfloat16_supported

MODEL_NAME = 'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit'
MAX_SEQ_LENGTH = 4096
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'

print(f'Loading FULL multimodal base model: {MODEL_NAME}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB\n')

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# Fix ClippableLinear for PEFT compatibility (PR #4807)
from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear

replaced = 0
for name, module in model.named_modules():
    if isinstance(module, Gemma4ClippableLinear):
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        replaced += 1

print(f'Replaced {replaced} Gemma4ClippableLinear -> nn.Linear')

# Apply language-only LoRA adapter
print(f'\nApplying text-only adapter from: {TEXT_ONLY_DIR}')
from peft import PeftModel
model = PeftModel.from_pretrained(model, TEXT_ONLY_DIR)

# Verify: only language params have LoRA, vision/audio untouched
lora_params = [n for n, p in model.named_parameters() if 'lora_' in n]
print(f'\nLoRA parameters loaded: {len(lora_params)}')
print(f'  language_model: {sum(1 for n in lora_params if "language_model" in n)}')
print(f'  vision_tower:   {sum(1 for n in lora_params if "vision_tower" in n)} (should be 0)')
print(f'  audio_tower:    {sum(1 for n in lora_params if "audio_tower" in n)} (should be 0)')

# Verify vision/audio towers exist in base model
has_vision = any('vision_tower' in n for n, _ in model.named_parameters())
has_audio = any('audio_tower' in n for n, _ in model.named_parameters())
has_projector = any('multi_modal_projector' in n for n, _ in model.named_parameters())
print(f'\nMultimodal towers present:')
print(f'  vision_tower:          {"YES" if has_vision else "MISSING!"}')
print(f'  audio_tower:           {"YES" if has_audio else "MISSING!"}')
print(f'  multi_modal_projector: {"YES" if has_projector else "MISSING!"}')

## 5. Quick Vision Inference Test (Pre-Merge)

Verify the model can process images before committing to the merge.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Download a test image (legal document / contract page)
test_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Constitution_of_the_United_States%2C_page_1.jpg/800px-Constitution_of_the_United_States%2C_page_1.jpg'
try:
    img_data = requests.get(test_url, timeout=10).content
    test_image = Image.open(BytesIO(img_data)).convert('RGB')
    print(f'Test image: {test_image.size} ({len(img_data) / 1024:.0f} KB)')
except Exception as e:
    print(f'Image download failed: {e}')
    # Create a simple test image
    test_image = Image.new('RGB', (384, 384), (200, 200, 200))
    print('Using placeholder image')

# Prepare prompt for VLM
FastVisionModel.for_inference(model)

from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained('unsloth/gemma-4-E4B-it-unsloth-bnb-4bit')

prompt = 'Analyze this document image. What type of legal document is this? Describe any key elements you can identify.'

messages = [
    {'role': 'user', 'content': [
        {'type': 'image', 'image': test_image},
        {'type': 'text', 'text': prompt},
    ]}
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        do_sample=True,
    )

response = processor.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'\n=== VLM Response (pre-merge) ===\n{response}')

## 6. Dequantize + Merge LoRA + Save Full Multimodal Model

**Critical**: Unlike the GRPO notebook which saves text-only, we save the **entire** model
including vision_tower, audio_tower, and multi_modal_projector.

Steps:
1. Reload base model fresh (4-bit)
2. Dequantize all Linear4bit → nn.Linear (BF16)
3. Apply language-only LoRA adapter
4. merge_and_unload() — merges LoRA into base BF16 weights
5. Save ALL tensors (language + vision + audio + projector)

In [ ]:
import torch, os, gc, json, subprocess
from safetensors.torch import save_file

# Ensure llama.cpp convert script is available
subprocess.run(['pip', 'install', 'safetensors', 'gguf', '--upgrade'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

CLEAN_DIR = 'gemma4-legal-vlm-merged'
TEXT_ONLY_DIR = 'gemma4-e4b-legal-text-only-adapter'
os.makedirs(CLEAN_DIR, exist_ok=True)

# Free memory from inference test (safe if model doesn't exist)
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# --- Step 1: Load fresh base model (4-bit) ---
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    model_name='unsloth/gemma-4-E4B-it-unsloth-bnb-4bit',
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)
print(f'Step 1/6: Base model loaded ({sum(p.numel() for p in model.parameters()):,} params)')

# --- Step 2: Fix ClippableLinear ---
from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
replaced = 0
for name, module in model.named_modules():
    if isinstance(module, Gemma4ClippableLinear):
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        replaced += 1
print(f'Step 2/6: Replaced {replaced} ClippableLinear -> nn.Linear')

# --- Step 3: Dequantize all Linear4bit -> nn.Linear (BF16) BEFORE adapter ---
import bitsandbytes as bnb
dequant_count = 0
for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        # Dequantize: NF4 -> BF16
        w = bnb.functional.dequantize_4bit(
            module.weight.data, module.weight.quant_state
        ).to(torch.bfloat16)
        new_linear = torch.nn.Linear(
            module.in_features, module.out_features,
            bias=module.bias is not None, dtype=torch.bfloat16, device=w.device
        )
        new_linear.weight.data = w
        if module.bias is not None:
            new_linear.bias.data = module.bias.data.to(torch.bfloat16)
        setattr(parent, parts[-1], new_linear)
        dequant_count += 1
        del w
print(f'Step 3/6: Dequantized {dequant_count} layers to BF16')

# --- Step 4: Apply language-only LoRA adapter ---
assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_model.safetensors'), \
    f'Text-only adapter not found at {TEXT_ONLY_DIR}/. Run Cell 3 first!'
assert os.path.exists(f'{TEXT_ONLY_DIR}/adapter_config.json'), \
    f'adapter_config.json missing from {TEXT_ONLY_DIR}/. Upload it alongside the safetensors file!'

from peft import PeftModel
model = PeftModel.from_pretrained(model, TEXT_ONLY_DIR)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Step 4/6: Adapter loaded ({trainable:,} trainable params)')

# --- Step 5: Merge LoRA into base weights ---
model = model.merge_and_unload()
total = sum(p.numel() for p in model.parameters())
print(f'Step 5/6: LoRA merged! {total:,} total params')

# Count multimodal params
vision_params = sum(p.numel() for n, p in model.named_parameters() if 'vision_tower' in n)
audio_params = sum(p.numel() for n, p in model.named_parameters() if 'audio_tower' in n)
proj_params = sum(p.numel() for n, p in model.named_parameters() if 'multi_modal_projector' in n)
lang_params = sum(p.numel() for n, p in model.named_parameters() if 'language_model' in n)
print(f'  language_model:        {lang_params:,} params')
print(f'  vision_tower:          {vision_params:,} params')
print(f'  audio_tower:           {audio_params:,} params')
print(f'  multi_modal_projector: {proj_params:,} params')

if vision_params == 0:
    print('\n  WARNING: vision_tower has 0 params — tower may not have been preserved!')
if audio_params == 0:
    print('\n  WARNING: audio_tower has 0 params — tower may not have been preserved!')

# --- Step 6: Save ALL tensors (full multimodal model) ---
sd = {}
for name, param in model.named_parameters():
    clean_name = name.replace('.base_layer.', '.').replace('base_model.model.', '')
    sd[clean_name] = param.data.to(torch.bfloat16).cpu()

save_file(sd, os.path.join(CLEAN_DIR, 'model.safetensors'))
tokenizer.save_pretrained(CLEAN_DIR)

# Save config (preserving multimodal architecture)
config = model.config if hasattr(model, 'config') else model.base_model.config
config_dict = config.to_dict() if hasattr(config, 'to_dict') else {}
with open(os.path.join(CLEAN_DIR, 'config.json'), 'w') as f:
    json.dump(config_dict, f, indent=2)

fsize = os.path.getsize(os.path.join(CLEAN_DIR, 'model.safetensors'))
print(f'Step 6/6: Saved {len(sd)} tensors ({fsize / 1024**3:.1f} GB) to {CLEAN_DIR}/')
print(f'\n=== Full multimodal model saved! ===')
print(f'  Vision tower: {"PRESERVED" if vision_params > 0 else "MISSING!"}')
print(f'  Audio tower:  {"PRESERVED" if audio_params > 0 else "MISSING!"}')
print(f'  Language:     GRPO LoRA MERGED')

## 7. Export Multimodal GGUF

Gemma 4 multimodal GGUF requires **two files**:
1. **Language model GGUF** — main weights (Q4_K_M quantized)
2. **mmproj GGUF** — multimodal projector (BF16, handles image encoding)

llama.cpp has day-one Gemma 4 multimodal support (April 2026).

In [ ]:
import os, subprocess

CLEAN_DIR = 'gemma4-legal-vlm-merged'
GGUF_DIR = 'gemma4-legal-vlm-gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

bf16_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# Step 1: Convert safetensors -> BF16 GGUF (full model)
print('Step 1/3: Converting merged BF16 safetensors -> BF16 GGUF...')
result = subprocess.run(
    ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
     '--outfile', bf16_path, '--outtype', 'bf16'],
    capture_output=True, text=True
)
if result.returncode == 0:
    fsize = os.path.getsize(bf16_path)
    print(f'  BF16 GGUF: {fsize / 1024**3:.1f} GB')
else:
    print(f'  ERROR: {result.stderr[:500]}')
    print('  Trying alternative conversion...')
    # Fallback: try with --model-type flag
    result = subprocess.run(
        ['python', 'llama.cpp/convert_hf_to_gguf.py', CLEAN_DIR,
         '--outfile', bf16_path, '--outtype', 'bf16', '--model-type', 'gemma4'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        fsize = os.path.getsize(bf16_path)
        print(f'  BF16 GGUF (alt): {fsize / 1024**3:.1f} GB')
    else:
        print(f'  Fallback also failed: {result.stderr[:500]}')

# Step 2: Quantize BF16 -> Q4_K_M
if os.path.exists(bf16_path):
    print('\nStep 2/3: Quantizing BF16 -> Q4_K_M...')
    # Build llama-quantize if not already built
    quantize_bin = 'llama.cpp/build/bin/llama-quantize'
    if not os.path.exists(quantize_bin):
        print('  Building llama-quantize...')
        subprocess.run(['cmake', '-B', 'llama.cpp/build', '-S', 'llama.cpp',
                       '-DCMAKE_BUILD_TYPE=Release'],
                      capture_output=True)
        subprocess.run(['cmake', '--build', 'llama.cpp/build', '--target', 'llama-quantize', '-j4'],
                      capture_output=True)
    
    result = subprocess.run(
        [quantize_bin, bf16_path, q4_path, 'Q4_K_M'],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(q4_path):
        fsize = os.path.getsize(q4_path)
        print(f'  Q4_K_M GGUF: {fsize / 1024**3:.1f} GB')
    else:
        print(f'  Quantization failed: {result.stderr[:500]}')

# Step 3: Extract multimodal projector -> mmproj GGUF
print('\nStep 3/3: Extracting multimodal projector...')
mmproj_script = 'llama.cpp/examples/llava/convert_image_encoder_to_gguf.py'
if not os.path.exists(mmproj_script):
    # Newer llama.cpp may have it at a different path
    mmproj_script = 'llama.cpp/tools/mtmd/convert_image_encoder_to_gguf.py'

if os.path.exists(mmproj_script):
    result = subprocess.run(
        ['python', mmproj_script, '--model_dir', CLEAN_DIR,
         '--output_path', mmproj_path],
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(mmproj_path):
        fsize = os.path.getsize(mmproj_path)
        print(f'  mmproj GGUF: {fsize / 1024**2:.0f} MB')
    else:
        print(f'  mmproj extraction failed: {result.stderr[:500]}')
        print('  NOTE: mmproj may already be included in the main GGUF for Gemma4')
        print('  Check: llama.cpp/docs/multimodal.md for current Gemma4 support')
else:
    print(f'  mmproj script not found at expected paths')
    print('  You may need to extract it manually or use Unsloth\'s export')

# Summary
print('\n=== GGUF Export Summary ===')
for f in [bf16_path, q4_path, mmproj_path]:
    if os.path.exists(f):
        print(f'  {os.path.basename(f)}: {os.path.getsize(f) / 1024**3:.2f} GB')
    else:
        print(f'  {os.path.basename(f)}: NOT CREATED')

## 8. Create Ollama Modelfile

Deploy as `gemma4-legal-vlm:latest` on Ollama with multimodal projector.

In [ ]:
import os

GGUF_DIR = 'gemma4-legal-vlm-gguf'
q4_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-Q4_K_M.gguf')
mmproj_path = os.path.join(GGUF_DIR, 'gemma4-legal-vlm-mmproj-BF16.gguf')

# Determine which GGUF to use
model_gguf = q4_path if os.path.exists(q4_path) else os.path.join(GGUF_DIR, 'gemma4-legal-vlm-BF16.gguf')

modelfile = f'''FROM {os.path.basename(model_gguf)}
'''

# Add mmproj if it exists as separate file
if os.path.exists(mmproj_path):
    modelfile += f'''PROJECTOR {os.path.basename(mmproj_path)}
'''

modelfile += '''PARAMETER temperature 0.3
PARAMETER num_predict 4096
PARAMETER num_ctx 32768
PARAMETER stop <end_of_turn>
PARAMETER stop <eos>

TEMPLATE """{{- range .Messages }}
{{- if eq .Role "system" }}<start_of_turn>system
{{ .Content }}<end_of_turn>
{{- else if eq .Role "user" }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}<end_of_turn>
{{- end }}
{{- end }}<start_of_turn>model
"""

SYSTEM """You are a legal AI assistant specialized in evidence analysis, case law research,
and legal document interpretation. You have been fine-tuned on legal reasoning tasks
including statutory interpretation, case analysis, evidence evaluation, and legal writing.

When analyzing images of legal documents, evidence photos, or exhibits:
- Identify the document type and key elements
- Extract relevant text, dates, signatures, and markings
- Note any anomalies, redactions, or chain-of-custody indicators
- Provide structured analysis suitable for legal proceedings
"""
'''

modelfile_path = os.path.join(GGUF_DIR, 'Modelfile')
with open(modelfile_path, 'w') as f:
    f.write(modelfile)

print('=== Modelfile Created ===')
print(modelfile)
print(f'\nSaved to: {modelfile_path}')
print(f'\n=== Deployment Commands ===')
print(f'cd {GGUF_DIR}')
print(f'ollama create gemma4-legal-vlm:latest -f Modelfile')
print(f'ollama run gemma4-legal-vlm:latest')

## 9. Video Frame Analysis Test

Gemma 4 processes video as a sequence of image frames.
- Configurable visual token budget: 70-1120 tokens per image
- Use low budget (70-140) for video = more frames, faster inference
- Max ~60 seconds of video at 1 FPS native

This cell demonstrates the frame extraction → batch analysis pattern
used by [mattsvlm](https://github.com/vast-data/mattsvlm).

In [ ]:
# Video frame analysis demo
# In production: extract frames with ffmpeg, send batch to Ollama

import subprocess, os, time
from PIL import Image

def extract_frames(video_path, fps=1, max_frames=30):
    """Extract frames from video at given FPS using ffmpeg."""
    frames_dir = 'video_frames'
    os.makedirs(frames_dir, exist_ok=True)
    
    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-frames:v', str(max_frames),
        '-q:v', '2',
        os.path.join(frames_dir, 'frame_%04d.jpg'),
        '-y'
    ]
    subprocess.run(cmd, capture_output=True)
    
    frames = sorted([
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.endswith('.jpg')
    ])
    return [Image.open(f).convert('RGB') for f in frames[:max_frames]]


def analyze_video_frames(frames, model, tokenizer, processor, prompt=None):
    """Analyze video frames as a batch of images."""
    if prompt is None:
        prompt = (
            'These are sequential frames from a video recording. '
            'Describe what is happening in this footage. '
            'Note any persons, actions, objects, or events relevant to a legal investigation.'
        )
    
    # Build message with multiple images
    content = [{'type': 'image', 'image': frame} for frame in frames]
    content.append({'type': 'text', 'text': prompt})
    
    messages = [{'role': 'user', 'content': content}]
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)
    
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
        )
    elapsed = time.time() - t0
    
    response = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response, elapsed


# Demo with synthetic frames (no video file needed)
print('=== Video Frame Analysis Demo ===')
print('Creating synthetic test frames...')

# Create 5 test frames with different content
test_frames = []
colors = [(200, 100, 100), (100, 200, 100), (100, 100, 200), (200, 200, 100), (200, 100, 200)]
for i, color in enumerate(colors):
    img = Image.new('RGB', (384, 384), color)
    test_frames.append(img)

print(f'Frames: {len(test_frames)} ({test_frames[0].size})')
print('\nNote: In production, use extract_frames() with real video files.')
print('Example: frames = extract_frames("evidence_recording.mp4", fps=1, max_frames=30)')
print('         response, elapsed = analyze_video_frames(frames, model, tokenizer, processor)')
print('\nFor Ollama deployment, send frames as base64 images array:')
print('  curl http://localhost:11434/api/generate -d \'{"model":"gemma4-legal-vlm","images":["<b64>","<b64>",...]}\'')
print('\nRecommended settings for video:')
print('  - Token budget: 70-140 per frame (speed over detail)')
print('  - FPS: 1-2 for surveillance, 0.5 for documents/static')
print('  - Max frames: 30 (Gemma4 handles up to ~60 at low budget)')

## 10. Package for Download

Creates zip files for Google Drive download → local deployment.

In [ ]:
import os, subprocess

GGUF_DIR = 'gemma4-legal-vlm-gguf'
CLEAN_DIR = 'gemma4-legal-vlm-merged'

# List all output files
print('=== Output Files ===')
for d in [GGUF_DIR, CLEAN_DIR]:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            fpath = os.path.join(d, f)
            if os.path.isfile(fpath):
                fsize = os.path.getsize(fpath)
                print(f'  {d}/{f}: {fsize / 1024**3:.2f} GB' if fsize > 1024**3 else f'  {d}/{f}: {fsize / 1024**2:.1f} MB')

# Optional: Copy to Google Drive for persistence
print('\n=== Save to Google Drive ===')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_DIR = '/content/drive/MyDrive/gemma4-legal-vlm-artifacts'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    
    # Copy GGUF files
    import shutil
    for f in os.listdir(GGUF_DIR):
        src = os.path.join(GGUF_DIR, f)
        dst = os.path.join(DRIVE_DIR, f)
        if os.path.isfile(src):
            print(f'  Copying {f}...')
            shutil.copy2(src, dst)
    
    print(f'\nSaved to Google Drive: {DRIVE_DIR}')
except Exception as e:
    print(f'Google Drive not available: {e}')
    print('Use files.download() to download directly')

print('\n=== Local Deployment ===')
print('1. Download GGUF files from Google Drive')
print('2. Place in: c:\\Users\\james\\Videos\\deeds-web-app\\trt_artifacts\\gemma4-legal-vlm\\')
print('3. Deploy to Ollama:')
print(f'   cd {GGUF_DIR}')
print('   ollama create gemma4-legal-vlm:latest -f Modelfile')
print('4. Test:')
print('   ollama run gemma4-legal-vlm:latest "Describe this image" --images test.jpg')
print('\n5. Update env.server.ts:')
print('   OLLAMA_VLM_MODEL=gemma4-legal-vlm:latest')

## 11. Upload to HF Hub (Optional)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
GGUF_DIR = 'gemma4-legal-vlm-gguf'
REPO_ID = 'Semaj90/gemma4-e4b-legal-vlm-GGUF'

print(f'Uploading VLM GGUF to {REPO_ID}...')
try:
    api.create_repo(REPO_ID, exist_ok=True)
    api.upload_folder(
        folder_path=GGUF_DIR,
        repo_id=REPO_ID,
        commit_message='Gemma 4 E4B Legal VLM — Q4_K_M GGUF + mmproj (vision re-attached)',
    )
    print(f'Uploaded! https://huggingface.co/{REPO_ID}')
except Exception as e:
    print(f'Upload failed: {e}')
    print(f'Manual: huggingface-cli upload {REPO_ID} {GGUF_DIR}')